# IOAI — 2025 National Human Vs Ai (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-human-vs-ai.zip', '/tmp/hva.zip')
    with zipfile.ZipFile('/tmp/hva.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Human vs AI — 모범답안2 (원대회 reference, 2개 서브태스크)
원대회 editorial 풀이 전체. **두 서브태스크**를 모두 풀어 원대회 제출 형식(`subtaskID,datapointID,answer`)을 만듭니다:
- **Subtask 1 (분류)**: 최소전처리 + 글자단위 `char_wb` TF-IDF(2–8) + `LinearSVC` 로 사람(0)/AI(1) 분류.
- **Subtask 2 (군집)**: 표준 전처리(불용어·구두점 제거·스테밍) + TF-IDF + `KMeans(4)` → TSNE 시각화 →
  각 군집 상위 단어로 토픽명 수기 부여(CRIME/SCIENCE/BUSINESS/RELIGION).

> ⚠️ Subtask 2(군집)는 train 에 토픽 정답이 없어 **로컬 자동채점 대상이 아닙니다**(플랫폼 채점). 이 노트북은 **실제 judge 제출용 `submission.csv`**(두 서브태스크)를 생성합니다. 로컬 자동채점되는 분류는 **모범답안** 참고.

In [ ]:
# 필요 패키지 자동 설치(DGX ioai-venv 는 nltk/seaborn 미포함, Colab 은 기본 제공) + nltk 리소스 다운로드
import subprocess, sys, importlib
def _ensure(pkg, mod=None):
    try: __import__(mod or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        importlib.invalidate_caches()
_ensure('nltk'); _ensure('seaborn')
import nltk
for r in ['stopwords', 'punkt', 'punkt_tab']:
    try: nltk.download(r, quiet=True)
    except Exception: pass

In [ ]:
import string

import pandas as pd
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

In [ ]:
seed = 42

# Data preparation

In [ ]:
def clean_df(df):
    return df

def clean_text(text: str):
    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    tokens = word_tokenize(text)

    # remove stopwords + digits
    stop_words = set(stopwords.words("english"))
    tokens = [word for word in tokens if word not in stop_words and not str.isdigit(word)]

    # stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]

    return ' '.join(tokens)

def prep_features(df: pd.DataFrame, for_unsupervised: bool):
    df = df.drop(["ID"], axis=1)

    # basic cleaning
    df["text"] = df["text"].apply(lambda s: s.lower().strip())

    if for_unsupervised:
        df["text"] = df["text"].apply(clean_text)
        return df["text"]
    
    if "label" in df:
        return df["text"], df["label"]
    return df["text"]

In [ ]:
df = pd.read_csv("data/train.csv")
df = clean_df(df)

X_supervised, y_supervised = prep_features(df, for_unsupervised=False)

In [ ]:
X_supervised.head()

# Model selection - subtask 1

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_supervised, y_supervised, test_size=0.33, random_state=seed)

In [ ]:
sup_vectorizer = TfidfVectorizer(min_df=2, max_df=0.95, analyzer='char_wb', ngram_range=(2, 8))

X_train = sup_vectorizer.fit_transform(X_train)
X_test = sup_vectorizer.transform(X_test)

In [ ]:
def evaluate(clf):
    cv = cross_val_score(clf, X_train, y_train, cv=3, n_jobs=-1)
    cv_score = cv.mean() - cv.std()

    clf.fit(X_train, y_train)
    score = f1_score(y_test, clf.predict(X_test))
    return cv_score, score

In [ ]:
svc = LinearSVC()

evaluate(svc)

In [ ]:
clf = svc
clf.fit(X_train, y_train)

# Clustering - subtask 2

In [ ]:
df_test = pd.read_csv("data/test.csv")
df_test = clean_df(df_test)

df_test_2 = df_test[df_test["subtaskID"] == 2].copy()
X_unsupervised = prep_features(df_test_2, for_unsupervised=True)

In [ ]:
vectorizer = TfidfVectorizer(min_df=5, max_df=0.8, stop_words='english')

X = vectorizer.fit_transform(X_unsupervised)

In [ ]:
km = KMeans(4, n_init=100, random_state=seed)
df_test_2["cluster"] = km.fit_predict(X)

In [ ]:
X_embedded = TSNE(n_components=2, init='random').fit_transform(X)

df_test_2["tsne_x"] = X_embedded[:, 0]
df_test_2["tsne_y"] = X_embedded[:, 1]

sns.scatterplot(x="tsne_x", y="tsne_y", hue="cluster", data=df_test_2)

In [ ]:
# print out samples from each cluster, to label them
terms = vectorizer.get_feature_names_out()
centroids = km.cluster_centers_
for cluster_idx in range(4):
    top_terms_idx = centroids[cluster_idx].argsort()[::-1][:10]

    top_terms = [terms[i] for i in top_terms_idx]
    print(f"cluster {cluster_idx}: {', '.join(top_terms)}")

In [ ]:
df_test_2["topic"] = df_test_2["cluster"].map({
    0: "CRIME",
    1: "SCIENCE",
    2: "BUSINESS",
    3: "RELIGION",
})

# Submission

In [ ]:
# subtask 1
df_test_1 = df_test[df_test["subtaskID"] == 1]

features = prep_features(df_test_1, for_unsupervised=False)
features = sup_vectorizer.transform(features)

subtask1 = clf.predict(features)

# subtask 2
subtask2 = df_test_2["topic"]

In [ ]:
def build_subtask(id, answer):
    df = df_test_1 if id == 1 else df_test_2
    return pd.DataFrame({
        "subtaskID": id, "datapointID": df["ID"], "answer": answer
    })

subtasks = [
    (1, subtask1),
    (2, subtask2)
]

submission = pd.concat([build_subtask(sid, subtask) for sid, subtask in subtasks], ignore_index=True)

In [ ]:
submission.head()

In [ ]:
submission.to_csv("submission_judge.csv", index=False)  # 실제 judge(2개 서브태스크)용

In [ ]:
# === 우리 채점기용: held-out(ID%5==0) 분류 예측 → submission.csv (ID,label) ===
# 원대회 reference 의 글자단위 char_wb 분류를 누수 없이(held-out 제외 학습) 재현해 로컬 채점 제출 생성.
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
_df = pd.read_csv('data/train.csv'); _df['text'] = _df['text'].fillna(''); _df['label'] = _df['label'].astype(float).astype(int)
_iv = _df['ID'] % 5 == 0
_tr, _va = _df[~_iv], _df[_iv]
_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 8), min_df=2, max_df=0.95)
_Xtr = _vec.fit_transform(_tr['text'].str.lower().str.strip())
_clf = LinearSVC().fit(_Xtr, _tr['label'])
_pred = _clf.predict(_vec.transform(_va['text'].str.lower().str.strip()))
pd.DataFrame({'ID': _va['ID'], 'label': _pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv (held-out 분류, 우리 채점기용)', len(_va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)